In [1]:
from pathlib import Path

import pandas as pd
from chemFilters.img_render import MolPlotter

[12:10:42] Initializing Normalizer


To run this notebook, install the chemFilters package:

```shell
python -m pip install git+https://github.com/David-Araripe/chemFilters.git
```

# Target Validation Screen
Render the compounds tested on this experiment

In [2]:
data_path = Path("../../data/target_validation/compound_data")
df = pd.read_csv(data_path / "chembl_data_validation_compounds.csv")

In [3]:
subset = df[
    [
        "smiles",
        "target_chembl_id",
        "molecule_chembl_id",
        "assay_type",
        "standard_type",
        "pchembl_value_mean",
        "pchembl_value_std",
        "pchembl_value_median",
        "doi",
    ]
].query("standard_type.isin(['IC50', 'Ki'])")

In [4]:
rest_of_data = pd.read_csv(data_path / "target_validation_cpds.csv", sep=";")

subset = subset.merge(
    rest_of_data,
    left_on=["molecule_chembl_id", "target_chembl_id"],
    right_on=["molecule_chembl_id", "target_chembl_id"],
    how="left",
)

In [5]:
plotter = MolPlotter(
    from_smi=True,
    size=(140, 80),
    bw=True,
    mol_font_size=8,
    bond_line_width=0.7,
    meanBondLength=1,
    scalingFactor=5,
    multipleBondOffset=0.18,
    additionalAtomLabelPadding=0.05,
)

smiles = subset[["smiles", "Compound Name"]].drop_duplicates()

for _, row in smiles.iterrows():
    txt = plotter.render_mol(row["smiles"], return_svg=True)
    with open(f"{row['Compound Name']}.svg", "w") as f:
        f.write(txt)

In [6]:
subset = subset.assign(
    img=lambda x: x["Compound Name"].apply(lambda y: f"![{y}](../figures/mol_structures/{y}.svg)")
)

In [7]:
format_decial_cases = lambda x: f"{x:.2f}"  # noqa: E731

print(
    subset[
        [
            "Target",
            "Compound Name",
            "img",
            "assay_type",
            "standard_type",
            "pchembl_value_mean",
            "pchembl_value_std",
        ]
    ]
    .assign(
        pchembl_value_mean=lambda x: x.pchembl_value_mean.apply(format_decial_cases),
        pchembl_value_std=lambda x: x.pchembl_value_std.apply(format_decial_cases),
    )
    .sort_values(["Target", "Compound Name", "assay_type"])
    .to_markdown(index=False)
)

| Target   | Compound Name   | img                                                         | assay_type   | standard_type   |   pchembl_value_mean |   pchembl_value_std |
|:---------|:----------------|:------------------------------------------------------------|:-------------|:----------------|---------------------:|--------------------:|
| ADORA1   | CPA             | ![CPA](../figures/mol_structures/CPA.svg)                   | B            | Ki              |                 8.41 |                0.46 |
| ADORA1   | CPA             | ![CPA](../figures/mol_structures/CPA.svg)                   | F            | IC50            |                 8.57 |                0    |
| ADORA1   | Capadenoson     | ![Capadenoson](../figures/mol_structures/Capadenoson.svg)   | B            | Ki              |                 8.85 |                0    |
| ADORA1   | DPCPX           | ![DPCPX](../figures/mol_structures/DPCPX.svg)               | B            | Ki              |                 8.5

# Compound Exploration Screen

In [8]:
import pandas as pd
from pathlib import Path
from chemFilters.img_render import MolPlotter

cpds = [
    "Z95680027",
    "Z318400112",
    "Z90308949",
    "824745",
    "1237561",
    "1249141",
    "1823372",
    "22755240",
    "27070328",
]
data_root = Path("../../").resolve() / "data/virtual_screening/vs_datasets/"


fnames = ["screened_adora1_computed_properties.csv", "screened_nr3c2_computed_properties.csv"]
df = pd.concat([pd.read_csv(data_root / f) for f in fnames], ignore_index=True)

In [9]:
df = df.query("id in @cpds")[["id", "hitSmiles", "target_id", "provider", "predictions"]].reset_index(
    drop=True
)

medchem_df = pd.DataFrame(
    [
        {
            "id": "Apararenone",
            "hitSmiles": "O=C1N(C2=CC=C(F)C=C2)C3=CC=C(NS(C)(=O)=O)C=C3OC1(C)C",
            "target_id": "NR3C2",
            "provider": "MedChemExpress",
            "predictions": "0.0",
            "chembl_id": "CHEMBL2181929",
        },
        {
            "id": "Benidipine",
            "hitSmiles": "O=C(C1=C(C)NC(C)=C(C(O[C@H]2CN(CC3=CC=CC=C3)CCC2)=O)[C@@H]1C4=CC=CC([N+]([O-])=O)=C4)OC",
            "target_id": "NR3C2",
            "provider": "MedChemExpress",
            "predictions": "0.0",
            "chembl_id": "CHEMBL2105555;CHEMBL3303980",
        },
        {
            "id": "MIPS521",
            "hitSmiles": "O=C(C1=C(N)SC=C1C2=CC(C(F)(F)F)=CC(C(F)(F)F)=C2)C3=CC=C(C=C3)Cl",
            "target_id": "ADORA1",
            "provider": "MedChemExpress",
            "predictions": "pKB=4.95",
            "chembl_id": "CHEMBL564720",
        },
    ],
)

df = (
    pd.concat([df, medchem_df], ignore_index=True)
    .replace({"enamine": "Enamine", "emolecules": "eMolecules"})
    .rename(columns={"hitSmiles": "smiles"})
    .sort_values(["target_id", "id"])
)

In [10]:
for _, row in df.iterrows():
    txt = plotter.render_mol(row["smiles"], return_svg=True)
    with open(f"{row['id']}.svg", "w") as f:
        f.write(txt)